In [ ]:
import pandas as pd
import scipy
import data_profiling
from data_profiling import ProfileReport
import numpy as np
import warnings
from utils import build_demand_hours

In [ ]:
warnings.filterwarnings("ignore")
uk_prices_2026 = pd.read_csv("data/raw/UK_elec_data_1y_Oct2026.csv", sep=";")
uk_prices_hist = pd.read_csv("data/raw/United_Kingdom.csv")
exchange_rates = pd.read_csv("data/raw/EURO_GBP_rates.csv")
day_ahead_demand = pd.read_csv("data/raw/dayahead_demand.csv") # forecasted before 09:20
day_ahead_wind = pd.read_csv("data/raw/dayahead_wind.csv") # forecasted before 09:20
brent_crude_oil = pd.read_csv("data/raw/DCOILBRENTEU.csv")
gas_SAP = pd.read_csv("data/raw/daily_SAP_gas_prices.csv")

In [ ]:
uk_prices_hist.head() 

In [ ]:
ukp = uk_prices_hist[["Datetime (Local)", "Price (EUR/MWhe)"]]
ukp["Datetime (Local)"] = pd.to_datetime(ukp["Datetime (Local)"])
ProfileReport(ukp)
;

In [ ]:
# converting to GBP

old_name = exchange_rates.columns.to_list()[-1]
exchange_rates = exchange_rates.rename(columns={old_name: "exchange_rate"})
exchange_rates["Date"] = pd.to_datetime(exchange_rates["Date"], format="%d %b %y")
exchange_rates.head()

In [ ]:
# filling in weekend exchange rates with the exchange rate on Friday 

full_range = pd.date_range(exchange_rates["Date"].min(), exchange_rates["Date"].max(), freq="D")

exchange_rates = (
    exchange_rates.set_index("Date")
        .sort_index()
        .reindex(full_range)
        .ffill()
        .rename_axis("Date")
        .reset_index()
)

In [ ]:
ukp["Date"] = ukp["Datetime (Local)"].dt.normalize()

# Merge the daily rate onto each row
ukp = ukp.merge(
    exchange_rates[["Date", "exchange_rate"]],
    on="Date",
    how="left",
)

# EUR/MWhe * GBP/EUR = GBP/MWhe
ukp["Price (GBP/MWhe)"] = ukp["Price (EUR/MWhe)"] / ukp["exchange_rate"]

In [ ]:
ukp["Hour"] = (
    ukp["Datetime (Local)"]
    .dt.tz_localize("Europe/London", nonexistent="NaT",ambiguous="NaT")
)

In [ ]:
ukp_hours = ukp[["Hour", "Price (GBP/MWhe)"]]
ukp_hours[-50:-45]

In [ ]:
day_ahead_demand.head() # UK local time

In [ ]:
# cleaning demand - removing extraneous columns, changing 2400 to 0000 next day

demand = day_ahead_demand[["TARGETDATE", "FORECASTDEMAND", "CP_ST_TIME", "CP_END_TIME", "FORECAST_TIMESTAMP"]]
demand_hours = build_demand_hours(demand, format="%Y-%m-%d")

In [ ]:
demand_hours.head()

In [ ]:
day_ahead_wind.head() # GMT

In [ ]:
wind = day_ahead_wind[["Datetime_GMT", "Capacity", "Incentive_forecast"]]
ProfileReport(wind)
;

In [ ]:
wind["Datetime_GMT"] = pd.to_datetime(wind["Datetime_GMT"])
wind_hours = (
    wind.set_index("Datetime_GMT")
        .resample("1h")["Incentive_forecast"]   
        .mean()
        .reset_index()
)
wind_hours["Hour"] = (
    wind_hours["Datetime_GMT"]
    .dt.tz_localize("UTC")
    .dt.tz_convert("Europe/London")
)
wind_hours = wind_hours.drop(columns = ["Datetime_GMT"])

In [ ]:
brent_crude_oil.head()

In [ ]:
brent = brent_crude_oil.copy()
brent["observation_date"] = pd.to_datetime(brent["observation_date"], format="%Y-%m-%d")
brent["DCOILBRENTEU"] = pd.to_numeric(brent["DCOILBRENTEU"], errors="coerce")  

# filling holidays and weekedns with data from the last trading day

brent = brent.set_index("observation_date").sort_index()
full_range = pd.date_range(brent.index.min(), brent.index.max(), freq="D")
brent = brent[["DCOILBRENTEU"]].reindex(full_range).ffill()
brent.index.name = "Date"

# 2-day lag: the row dated D holds the price from D-2 (e.g. 11/10/26 holds 09/10/26)
# this is so we have the data that would be available to us at 9:20 UK time 
# technically possible that we could do only 1 day lag but couldn't find suitable dataset
 
brent_lag = brent["DCOILBRENTEU"].shift(2, freq="D")

# creating brent_hours so can merge with other dataframes

hours = pd.DataFrame({"hod": range(24)})
brent_hours = brent_lag.reset_index().merge(hours, how="cross")
brent_hours["Hour"] = brent_hours["Date"] + pd.to_timedelta(brent_hours["hod"], unit="h")
brent_hours = brent_hours.sort_values("Hour").reset_index(drop=True)[["Hour", "DCOILBRENTEU"]]
brent_hours["Hour"] = brent_hours["Hour"].dt.tz_localize("Europe/London", nonexistent="NaT",ambiguous="NaT")

In [ ]:
brent_hours.head()

In [ ]:
gas_SAP.head()

In [ ]:
gas = gas_SAP.copy()
gas["Date"] = pd.to_datetime(gas["Date"], format='%Y-%m-%d')   # add format=... or dayfirst=True if needed
gas["SAP actual day"] = pd.to_numeric(gas["SAP actual day"], errors="coerce")

gas = gas.drop_duplicates("Date").set_index("Date").sort_index()
full_range = pd.date_range(gas.index.min(), gas.index.max(), freq="D")
gas = gas[["SAP actual day"]].reindex(full_range).ffill()
gas.index.name = "Date"

# 2-day lag: the row dated D holds the SAP of gas day D-2
# (gas day D-2 ends 05:00 on D-1, before the day-ahead auction for D)
gas_lag = gas["SAP actual day"].shift(2, freq="D").rename("sap_lag2d").rename_axis("Date")

# Long format: one row per (Date, hour of day), 24 rows per day
hours = pd.DataFrame({"hod": range(24)})
gas_hours = gas_lag.reset_index().merge(hours, how="cross")
gas_hours["Hour"] = gas_hours["Date"] + pd.to_timedelta(gas_hours["hod"], unit="h")
gas_hours = gas_hours.sort_values("Hour").reset_index(drop=True)[["Hour", "sap_lag2d"]]
gas_hours["Hour"] = gas_hours["Hour"].dt.tz_localize("Europe/London", nonexistent="NaT",ambiguous="NaT")

In [ ]:
gas_hours.head()

In [ ]:
df = (
    wind_hours
        .merge(demand_hours, on="Hour", how="inner")
        .merge(brent_hours, on="Hour", how="inner" )
        .merge(gas_hours, on="Hour", how="inner")
        .merge(ukp_hours, on="Hour", how="inner")
)

In [ ]:
ProfileReport(df)
;

In [ ]:
df.to_parquet("data/processed/base_EPEX_df.parquet")